# Normalize Manual expert annotation names in the dataset

In [1]:
from pathlib import Path

folder = Path("/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented")


In [3]:
 [file for file in folder.rglob("*") if file.is_file()]

[PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Poznámky.odoc'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/test.txt'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/Lung.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/Spleen.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/Liver.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/Kidney.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/Tx017D_Art.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/labelmap.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx017D_Art/Artery.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx018D_Art/kidney.nii.gz'),
 PosixPath('/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented/Tx018D_Art/spleen.nii.gz'),
 PosixPath('/home/mjiri

In [16]:
from pathlib import Path
import re
import shutil

# === NASTAVENÍ ===
root = Path("/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented")
output_dir = Path("/home/mjirik/mnt/nas-bmc3_ct/pilsen_pigs_segmented_renamed")  # nebo None
ignore_background = True
flatten_output = True  # pokud True, všechny soubory půjdou do jednoho adresáře
dry_run = True  # True = pouze výpis, False = reálné kopírování/přejmenování
dry_run = False

# === MAPOVÁNÍ STRUKTUR ===
structure_map = {
    "artery": "Arteries",
    "arteries": "Arteries",
    "liver": "Liver",
    "kidney": "Kidney",
    "spleen": "Spleen",
    "lung": "Lung",
    "background": "Background",
}

# === FUNKCE NA NORMALIZACI ID ===
def normalize_id(pig_id: str) -> str:
    """Z Tx17D udělá Tx017D, z H3 udělá H03."""
    if pig_id.startswith("Tx"):
        m = re.search(r"Tx(\d{1,3})D", pig_id)
        if m:
            num = int(m.group(1))
            return f"Tx{num:03d}D"
    elif pig_id.startswith("H"):
        m = re.search(r"H(\d{1,3})", pig_id)
        if m:
            num = int(m.group(1))
            return f"H{num:03d}"  # nebo {num:03d}, pokud chceš i H### formát
    return pig_id


# === HLAVNÍ SMYČKA ===
for path in root.rglob("*.*"):
    if path.suffix not in [".nrrd", ".gz", ".nii"]:
        continue


    # 0️⃣ přeskoč velké .nii.gz soubory (typicky zdrojové CT)
    try:
        if path.suffixes == [".nii", ".gz"] and path.stat().st_size > 1_000_000:
            # > 1 MB → pravděpodobně CT objem, ne anotace
            continue
    except (OSError, PermissionError):
        # pokud by stat() selhal (např. chybějící přístup), přeskočíme
        continue

    # 1️⃣ identifikátor prasete
    m = re.search(r"(Tx\d{1,3}D|H\d{1,3})", str(path))
    if not m:
        continue
    pig_id = normalize_id(m.group(1))

    # 2️⃣ fáze
    phase = "Ven" if "Ven" in path.name or "Ven" in str(path) else "Art"

    # 3️⃣ struktura
    lower_name = path.name.lower()
    structure = None
    for key, val in structure_map.items():
        if key in lower_name:
            structure = val
            break
    if structure is None:
        if "segmentation" in lower_name:
            structure = "Arteries"
        else:
            continue
    if ignore_background and structure == "Background":
        continue

    # 4️⃣ přípona
    suffix = ".nii.gz" if path.name.endswith(".nii.gz") else path.suffix

    # 5️⃣ expert
    expert_suffix = "expert_2" if any("Pig" in part for part in path.parts) else "expert"

    # 6️⃣ nový název
    new_name = f"{pig_id}_{phase}-{structure}-{expert_suffix}{suffix}"

    # 7️⃣ cílová cesta
    if output_dir is not None:
        rel_path = path.relative_to(root).parent
        if flatten_output:
            new_path = output_dir / new_name
        else:
            new_path = output_dir / rel_path / new_name
    else:
        new_path = path.parent / new_name

    # === AKCE ===
    print(f"{path.relative_to(root)} → {new_path.relative_to(root if output_dir is None else output_dir)}")

    if not dry_run:
        new_path.parent.mkdir(parents=True, exist_ok=True)
        if output_dir is not None:
            shutil.copyfile(path, new_path)  # kopírování do nové složky
        else:
            shutil.move(path, new_path)  # přejmenování na místě


Tx017D_Art/Lung.nii.gz → Tx017D_Art-Lung-expert.nii.gz
Tx017D_Art/Spleen.nii.gz → Tx017D_Art-Spleen-expert.nii.gz
Tx017D_Art/Liver.nii.gz → Tx017D_Art-Liver-expert.nii.gz
Tx017D_Art/Kidney.nii.gz → Tx017D_Art-Kidney-expert.nii.gz
Tx017D_Art/Artery.nii.gz → Tx017D_Art-Arteries-expert.nii.gz
Tx018D_Art/kidney.nii.gz → Tx018D_Art-Kidney-expert.nii.gz
Tx018D_Art/spleen.nii.gz → Tx018D_Art-Spleen-expert.nii.gz
Tx018D_Art/lung.nii.gz → Tx018D_Art-Lung-expert.nii.gz
Tx018D_Art/artery.nii.gz → Tx018D_Art-Arteries-expert.nii.gz
Tx019D_Ven/artery.nii.gz → Tx019D_Ven-Arteries-expert.nii.gz
Tx023D_Ven/artery.nii.gz → Tx023D_Ven-Arteries-expert.nii.gz
Tx024D_Art/artery.nii.gz → Tx024D_Art-Arteries-expert.nii.gz
Tx025D_Ven/artery.nii.gz → Tx025D_Ven-Arteries-expert.nii.gz
Tx025D_Art/artery.nii.gz → Tx025D_Art-Arteries-expert.nii.gz
Tx021D_Ven/artery.nii.gz → Tx021D_Ven-Arteries-expert.nii.gz
Tx032D_Art/Segmentation.seg.nrrd → Tx032D_Art-Arteries-expert.nrrd
Tx033D_Art/Segmentation preview.seg.nrrd →